# 03 (full data) · LightGBM matcher and the submission files

The method of `03_lightgbm_matcher.ipynb`, the best one on the 500K sample (LightGBM + one owner:
0.974 vs 0.935 for the cosine rule), trained and applied at full scale:

1. **Train** on the full-size train buckets from `02_full_e5_buckets.ipynb`:
   * `TRAIN_S1_PER_COUNTRY` random train S1 per country, with their top `CANDIDATES_PER_S1` e5
     candidates plus the extra candidates 02 added (reverse search, address key, name 3-grams)
   * the features that compare a candidate across S1s use every S1 of the country, exactly as on test
2. **Cross-validate** (grouped by S1), tune the cutoff t for the challenge metric, and score. True
   matches that the buckets missed count as misses here, so this is a realistic estimate of the
   leaderboard score.
3. **Final model** on all training pairs.
4. **Predict** every test bucket. Each S2/S3 record goes to at most one S1, and pairs with
   p ≥ t are kept. `COUNTRY_THRESHOLDS` can set a stricter t for one country (see France below).
5. **Write** `output/matching_results.tsv` and `output/candidate_pairs.tsv`, in the format of the
   problem statement, with one row for every S1 in `test_source1.tsv` (France included). The
   candidates are exactly the pairs the model scored, and the matches are a subset of them.
6. **Check** both files against the submission rules, and run `validate_submission.py` if it is
   attached.

**Input:** the output of `02_full_e5_buckets.ipynb` (`embeddings_full/.../buckets/`), 01's
`processed/`, and the competition data (`train_ground_truth.tsv`, `test/test_source*.tsv`).

**Output** (`submission/`):

| File | Content |
|---|---|
| `output/matching_results.tsv` | final matches: the file to upload to the leaderboard |
| `output/candidate_pairs.tsv` | the candidate set the model scored (for the final zip) |
| `model/lgbm_matcher.txt`, `model/matcher_config.json` | final model, features, threshold, scores |
| `model/metric_f05.csv`, `model/feature_importance.csv` | cross-validation scores and feature importance |
| `model/oof_pairs.parquet` | every training pair with its cross-validation probability |
| `test_probabilities/<country>.parquet` | probability of every test pair (to try another threshold without recomputing) |

**Run on Kaggle:** attach the competition data, 01's output and 02 (full)'s output, then Run all.
Most of the work is CPU (text comparisons). My estimate is 45-90 minutes. LightGBM uses the GPU
only if Kaggle's LightGBM build supports it.

Settings are in the first code cell and can also be set as environment variables: `BUCKET_DIR`,
`PROCESSED_DIR`, `DATA_ROOT`, `SUBMISSION_DIR`, `CANDIDATES_PER_S1`, `TRAIN_S1_PER_COUNTRY`,
`LGBM_DEVICE`, `VALIDATOR`.

In [1]:
import collections
import csv
import gc
import glob
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

for pkg in ("lightgbm", "rapidfuzz", "anyascii"):   # MIT, MIT, ISC
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import lightgbm as lgb
from rapidfuzz import fuzz
from anyascii import anyascii
from rapidfuzz.distance import JaroWinkler

CANDIDATES_PER_S1 = int(os.environ.get("CANDIDATES_PER_S1", 30))          # top e5 candidates judged per S1 (30 beat 20 on the test bed)
USE_EXTRA_CANDIDATES = os.environ.get("USE_EXTRA_CANDIDATES", "true") == "true"  # also judge 02's extra candidates (if present)
TRAIN_S1_PER_COUNTRY = int(os.environ.get("TRAIN_S1_PER_COUNTRY", 200_000))  # train S1 used for learning, per country
N_FOLDS = 3
SEED = 42
LGBM_DEVICE = os.environ.get("LGBM_DEVICE", "auto")                        # auto | cpu | gpu | cuda
CROSS_COUNTRY_CHECK = True                                                  # train on one country, score the other
PAIRS_PER_CHUNK = 2_000_000                                                 # pairs featurized at a time (memory)
FEATURE_SET = os.environ.get("FEATURE_SET", "all")                          # all | base (the 44 of notebook 03)
COUNTRY_THRESHOLDS = {}   # optional stricter cutoff per test country, e.g. {"France": 0.9}; others use the tuned t
T0 = time.time()


def search(patterns, roots):
    """First existing file matching any glob pattern under any root (shortest path wins); skips __MACOSX."""
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for pat in patterns for h in glob.glob(os.path.join(root, pat), recursive=True) if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    return None


KAGGLE = os.path.isdir("/kaggle/working")
LOCAL_DATA = str(Path.cwd().parent / "Data" / "student_resource")
if os.environ.get("BUCKET_DIR"):
    BUCKET_DIR = Path(os.environ["BUCKET_DIR"])
else:
    hit = search(["**/buckets/train_*.parquet", "**/buckets/test_*.parquet"],
                 ["embeddings_full", "/kaggle/working/embeddings_full", "/kaggle/input"])
    assert hit is not None, "no buckets found: run 02_full_e5_buckets.ipynb (or attach its output) or set BUCKET_DIR"
    BUCKET_DIR = hit.parent
MANIFEST = json.loads((BUCKET_DIR.parent / "manifest.json").read_text(encoding="utf-8")) \
    if (BUCKET_DIR.parent / "manifest.json").exists() else {}
proc = search(["train_s1_*.parquet", "**/train_s1_*.parquet"],
              [os.environ.get("PROCESSED_DIR", ""), MANIFEST.get("settings", {}).get("processed_dir", ""), "processed",
               "/kaggle/working/processed", "/kaggle/input"])
assert proc is not None, "processed/ not found: attach 01's output or set PROCESSED_DIR"
PROCESSED_DIR = proc.parent
DATA_ROOTS = [os.environ.get("DATA_ROOT", ""), "/kaggle/input", LOCAL_DATA]
GT_FILE = search(["**/train_ground_truth.tsv"], DATA_ROOTS)
TEST_S1_FILE = search(["**/test_source1.tsv"], DATA_ROOTS)
assert GT_FILE is not None and TEST_S1_FILE is not None, "competition data not found: attach it or set DATA_ROOT"
TEST_DIR = TEST_S1_FILE.parent
VALIDATOR = Path(os.environ["VALIDATOR"]) if os.environ.get("VALIDATOR") else \
    search(["**/validate_submission.py"], ["/kaggle/input", LOCAL_DATA])
OUT_DIR = Path(os.environ.get("SUBMISSION_DIR") or ("/kaggle/working/submission" if KAGGLE else "submission"))
for sub in ("output", "model", "test_probabilities"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
print(f"buckets: {BUCKET_DIR}\nprocessed: {PROCESSED_DIR}\nground truth: {GT_FILE}\ntest data: {TEST_DIR}\n"
      f"validator: {VALIDATOR}\noutput: {OUT_DIR}")
BUCKETS = sorted(p.stem for p in BUCKET_DIR.glob("*_*.parquet"))
TRAIN_COUNTRIES = [b.split("_", 1)[1] for b in BUCKETS if b.startswith("train_")]
TEST_COUNTRIES = [b.split("_", 1)[1] for b in BUCKETS if b.startswith("test_")]
print(f"train buckets: {TRAIN_COUNTRIES} | test buckets: {TEST_COUNTRIES}")
assert TRAIN_COUNTRIES and TEST_COUNTRIES, "both train and test buckets are needed (run 02 full with SPLITS=train,test)"


def lgbm_device():
    """GPU if this LightGBM build supports it (tried on a tiny dataset), else CPU."""
    if LGBM_DEVICE != "auto":
        return LGBM_DEVICE
    if shutil.which("nvidia-smi") is None:
        return "cpu"
    x = np.random.default_rng(0).random((2000, 4))
    for device in ("cuda", "gpu"):
        try:
            lgb.train({"objective": "binary", "device_type": device, "verbose": -1},
                      lgb.Dataset(x, (x[:, 0] > 0.5).astype(int)), num_boost_round=2)
            return device
        except Exception:
            pass
    print("this LightGBM build has no GPU support, training on the CPU (the 'Fatal' lines above come from that test)")
    return "cpu"


DEVICE = lgbm_device()
PARAMS = {"objective": "binary", "learning_rate": 0.1, "num_leaves": 63, "min_data_in_leaf": 100,
          "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
          "max_bin": 255 if DEVICE == "cpu" else 63, "device_type": DEVICE, "seed": SEED, "verbose": -1}
print(f"LightGBM {lgb.__version__} on {DEVICE}; {CANDIDATES_PER_S1} candidates per S1; "
      f"{TRAIN_S1_PER_COUNTRY:,} train S1 per country; {N_FOLDS} folds")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 5.6 MB/s eta 0:00:00
buckets: /kaggle/input/datasets/trezwakaxa/embeddings-2/embeddings_full/multilingual-e5-small/buckets
processed: /kaggle/input/datasets/trezwakaxa/preprocessed-data-2/processed
ground truth: /kaggle/input/datasets/trezwakaxa/all-data/train_ground_truth.tsv
test data: /kaggle/input/datasets/trezwakaxa/all-data
validator: None
output: /kaggle/working/submission
train buckets: ['India', 'US'] | test buckets: ['France', 'India', 'US']


[LightGBM] [Fatal] CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


LightGBM 4.6.0 on gpu; 30 candidates per S1; 200,000 train S1 per country; 3 folds


## Challenge metric: macro F0.5 per S1
For every S1, F0.5 over its predicted vs true S2/S3 IDs, averaged over all S1. An S1 with no true
match scores 1.0 for an empty prediction and 0.0 otherwise.

In [ ]:
def f05(predicted, true):
    """Challenge F0.5 for one S1: predicted and true are collections of S2/S3 entity IDs."""
    predicted, true = set(predicted), set(true)
    if not true:
        return 1.0 if not predicted else 0.0
    tp = len(predicted & true)
    if tp == 0:
        return 0.0
    precision, recall = tp / len(predicted), tp / len(true)
    return 1.25 * precision * recall / (0.25 * precision + recall)


def macro_f05(pred_by_s1, true_by_s1, s1_ids):
    """Mean challenge F0.5 over s1_ids (S1 missing from pred_by_s1 = empty prediction)."""
    return float(np.mean([f05(pred_by_s1.get(s, ()), true_by_s1.get(s, ())) for s in s1_ids]))


assert round(f05(["S2-00047", "S2-00193", "S3-00812"], ["S2-00047", "S3-00812"]), 3) == 0.714
print("challenge metric OK (problem-statement example = 0.714)")

## Records and features
The 44 features of notebook 03, 13 that separate lookalikes and 4 that say which search found the
pair (61 in total), computed from 01's processed fields and the buckets:

| Group | Features |
|---|---|
| Embedding | rank, retrieval score, cosine; gap to the S1's best candidate and to the next one; the S1's best and top-5 cosine |
| Other S1s | how many S1 buckets hold this candidate, whether this S1 is its best S1, and the score margin to its other best S1 |
| Name | edit-distance ratios, token-set/sort ratios, Jaro-Winkler, transliterated-name ratios, compact name/domain ratio, phonetic code, legal form, lengths |
| Address | ratios on the cleaned and raw address, shared street tokens, shared numbers, postcode, house number, city, state |
| Candidate flags | S2 or S3, native script, web name, empty address, honorific, landmark |
| **Street** (new) | the street name (the address part with the house number, minus numbers, street words like rue/avenue/road and stop words) compared by token-set and plain ratio; the house number (leading zeros removed) |
| **Word rarity** (new) | name and address word overlap weighted by how rare each word is among the S1 of that country (IDF), the share of the S1's weight found in the candidate, and the rarest unmatched word on each side |
| **Name frequency** (new) | how many S1 per 100,000 in that country have the S1's name, and the candidate's name |
| **Found by** (with 02's extra sources) | rank in the e5 search, rank of this S1 among the candidate's best S1 (reverse search), same house number + street word, rank among the S1's most similar names by 3-grams; empty when that search did not find the pair |

Why the new ones: France's S1 sit in about 15 cities and their names are mostly a city plus a
generic word ("Bordeaux Club" is one of the most common). Lookalikes there share the city, house
number, a generic word and the legal form, but not the street or the rare name word. Rarity is
measured per country (only from unlabeled S1 records), so "club" weighs little in France.

Country is not a feature, so the model applies to France. Where a field is empty on either side,
the feature is left empty (NaN) and LightGBM treats it as unknown.

**Indian scripts:** 01's cleaning strips combining marks, which deletes the vowel signs of Indian
scripts, so its `name_norm`, `name_lat` and `addr_norm` are garbled for such records. For every
name or address with a native script, all comparisons here use the original text transliterated
with `anyascii` instead ("సన్ ఎనర్జీ లిమిటెడ్" -> "sn enrji limited"), as 02 (full) does for the
embeddings. Latin-script records are unchanged.

In [ ]:
try:
    from rapidfuzz.process import cpdist   # rapidfuzz >= 3.6: many pairs at once on all CPU cores
except ImportError:
    cpdist = None

STR_COLS = ["business_name", "business_address", "name_norm", "addr_norm", "core_name", "name_lat", "legal_form",
            "domain_stem", "phonetic_code", "street_tokens", "house_no", "city", "state_code"]
FLAG_COLS = ["has_honorific", "empty_addr", "landmark_flag"]
TAGS = ["tag_native_script", "tag_web_name", "tag_empty_address", "tag_honorific", "tag_landmark"]
# a name has a native script when it has a character outside Latin, Latin-1/Extended and general punctuation
_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
RE_NATIVE = re.compile("[^" + "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES) + "]")
RE_HANDLE = re.compile(r"^\s*[@#]\S+\s*$")
RE_NUM = re.compile(r"\d+")
RE_POSTCODE = re.compile(r"(?<!\d)\d{5,6}(?!\d)")
RE_NOT_ALNUM = re.compile(r"[^0-9a-z]+")
FEATURES = ["rank", "score", "cosine", "s1_best_cosine", "gap_to_best_score", "gap_to_next_score", "s1_top5_cosine",
            "cand_n_s1", "cand_rank", "cand_margin", "name_ratio", "name_token_set", "name_token_sort", "name_partial",
            "core_ratio", "core_jaro_winkler", "latin_ratio", "latin_token_set", "raw_name_token_set", "compact_ratio",
            "compact_partial", "phonetic_same", "legal_form_same", "s1_name_len", "cand_name_len", "addr_ratio",
            "addr_token_set", "raw_addr_token_set", "raw_addr_partial", "street_jaccard", "street_cover",
            "numbers_jaccard", "numbers_cover", "postcode_same", "house_no_same", "city_same", "state_same",
            "cand_addr_len", "cand_is_s3", "cand_native_script", "cand_web_name", "cand_empty_address",
            "cand_honorific", "cand_landmark"]
BASE_FEATURES = list(FEATURES)
FEATURES = FEATURES + ["name_idf_jaccard", "name_idf_cover", "name_idf_miss_s1", "name_idf_miss_cand", "addr_idf_jaccard",
                       "addr_idf_cover", "addr_idf_miss_s1", "addr_idf_miss_cand", "street_name_token_set",
                       "street_name_ratio", "house_number_same", "s1_name_freq", "cand_name_freq"]
SOURCE_FEATURES = ["e5_rank", "rev_rank", "addr_key", "name_rank"]              # written by 02 with extra sources
FEATURES = FEATURES + SOURCE_FEATURES
if FEATURE_SET == "base":
    FEATURES = BASE_FEATURES
_ACCENTS = {}
for _cp in range(0xC0, 0x250):
    _base = unicodedata.normalize("NFKD", chr(_cp))[0]
    if _base.isascii() and _base != chr(_cp):
        _ACCENTS[_cp] = _base
FOLD = str.maketrans(_ACCENTS)
RE_WORD = re.compile(r"\w+")
STREET_TYPES = frozenset("rue avenue av ave boulevard bd blvd boul chemin che place pl route rte allee impasse imp quai "
                         "cours crs square faubourg fbg road rd street st lane ln drive dr way court ct circle cir "
                         "highway hwy parkway pkwy terrace marg".split())
STREET_STOP = STREET_TYPES | frozenset("r ch all sq bis ter de du des la le les l d of the and et en sur suite ste unit "
                                       "apt apartment building bldg floor fl po box door no nr n number h hno flat shop "
                                       "plot near opp".split())


def fold_text(s):
    """Lower case with Latin accents removed (other scripts unchanged)."""
    return s.lower().translate(FOLD)


def words(s):
    """Folded word tokens; numbers without leading zeros ('0058' -> '58')."""
    return [(w.lstrip("0") or "0") if w.isdigit() else w for w in RE_WORD.findall(fold_text(s))]


def street_of(address):
    """(street name, house number): the first comma part with a number or a street word, minus numbers, street
    words and stop words. '15 bis Rue de Barbieux, Roubaix' -> ('barbieux', '15')."""
    house = ""
    for part in address.split(","):
        w = words(part)
        nums = [x for x in w if x.isdigit()]
        if nums and not house:
            house = nums[0]
        if nums or any(x in STREET_TYPES for x in w):
            name = " ".join(x for x in w if not x.isdigit() and x not in STREET_STOP and len(x) > 1)
            if name:
                return name, house
    return "", house


def country_stats(split, country):
    """Word rarity (IDF) in the names and addresses of all S1 of one split and country, and how many S1 share each name."""
    s1 = pd.read_parquet(PROCESSED_DIR / f"{split}_s1_{country}.parquet", columns=["core_name", "business_address"])
    names = [frozenset(words(x)) for x in s1["core_name"].fillna("").astype(str)]
    addrs = [frozenset(words(x)) for x in s1["business_address"].fillna("").astype(str)]
    n = len(s1)

    def idf(sets):
        df = collections.Counter(t for s in sets for t in s)
        return {t: math.log((n + 1) / (c + 1)) + 1.0 for t, c in df.items()}

    return {"name_idf": idf(names), "addr_idf": idf(addrs), "unseen_idf": math.log(n + 1) + 1.0, "n": n,
            "name_freq": collections.Counter(" ".join(sorted(s)) for s in names)}


def weighted_overlap(a, b, idf, unseen):
    """IDF-weighted Jaccard, share of the S1's weight found in the candidate, and the rarest unmatched word on each
    side, for aligned arrays of word sets (NaN when either side is empty)."""
    out = np.full((4, len(a)), np.nan, np.float32)
    g = idf.get
    for i, (x, y) in enumerate(zip(a, b)):
        if not x or not y:
            continue
        inter = x & y
        wi = sum(g(t, unseen) for t in inter)
        wx = sum(g(t, unseen) for t in x)
        wy = sum(g(t, unseen) for t in y)
        out[0, i] = wi / (wx + wy - wi)
        out[1, i] = wi / wx
        out[2, i] = max((g(t, unseen) for t in x - inter), default=0.0)
        out[3, i] = max((g(t, unseen) for t in y - inter), default=0.0)
    return out


def load_records(split, country, ids):
    """Fields of the given entity IDs from processed/<split>_s{1,2,3}_<country>.parquet, plus per-record helpers."""
    wanted = pa.array(sorted(ids), pa.large_string())
    frames = []
    for src in ("s1", "s2", "s3"):
        f = PROCESSED_DIR / f"{split}_{src}_{country}.parquet"
        if not f.exists():
            continue
        names = set(pq.read_schema(f).names)
        t = pq.read_table(f, columns=["entity_id"] + [c for c in STR_COLS + FLAG_COLS if c in names])
        t = t.filter(pc.is_in(t["entity_id"].cast(pa.large_string()), value_set=wanted))
        frames.append(t.to_pandas())
    ent = pd.concat(frames, ignore_index=True).drop_duplicates("entity_id").set_index("entity_id")
    for c in STR_COLS:
        ent[c] = ent[c].fillna("").astype(str) if c in ent else ""
    for c in FLAG_COLS:
        ent[c] = pd.to_numeric(ent[c], errors="coerce").fillna(0).astype(np.int8) if c in ent else 0
    name = ent["business_name"].tolist()
    ent["tag_native_script"] = [RE_NATIVE.search(x) is not None for x in name]
    # native-script names/addresses: every comparison uses the original text transliterated (01's is garbled)
    native_name = ent["tag_native_script"].to_numpy()
    if native_name.any():
        lat = [" ".join(anyascii(x).lower().split()) for x in ent.loc[native_name, "business_name"]]
        for c in ("business_name", "name_norm", "core_name", "name_lat"):
            ent.loc[native_name, c] = lat
    native_addr = np.array([RE_NATIVE.search(x) is not None for x in ent["business_address"]])
    if native_addr.any():
        lat = [anyascii(x) for x in ent.loc[native_addr, "business_address"]]
        ent.loc[native_addr, "business_address"] = lat
        ent.loc[native_addr, "addr_norm"] = [" ".join(x.lower().replace(",", " ").split()) for x in lat]
    ent["tag_web_name"] = (ent["domain_stem"] != "").to_numpy() | np.array([RE_HANDLE.match(x) is not None for x in name])
    ent["tag_empty_address"] = (ent["empty_addr"] == 1) | (ent["addr_norm"] == "")
    ent["tag_honorific"] = ent["has_honorific"] == 1
    ent["tag_landmark"] = ent["landmark_flag"] == 1
    ent["raw_name"] = ent["business_name"].str.lower()
    ent["raw_addr"] = ent["business_address"].str.lower()
    ent["compact"] = [RE_NOT_ALNUM.sub("", (d or l or n).lower())
                      for d, l, n in zip(ent["domain_stem"], ent["name_lat"], ent["name_norm"])]
    ent["name_words"] = [frozenset(words(x)) for x in ent["core_name"]]
    ent["addr_words"] = [frozenset(words(x)) for x in ent["business_address"]]
    ent["numbers"] = [frozenset(x for x in w if x.isdigit()) for w in ent["addr_words"]]   # '0058' == '58'
    streets = [street_of(x) for x in ent["business_address"]]
    ent["street_name"] = [s for s, _ in streets]
    ent["house_number"] = [h for _, h in streets]
    ent["name_key"] = [" ".join(sorted(w)) for w in ent["name_words"]]
    ent["postcode"] = [(RE_POSTCODE.findall(a) or [""])[-1] for a in ent["raw_addr"]]
    ent["street_set"] = [frozenset(s.split()) for s in ent["street_tokens"]]
    return ent.drop(columns=["business_name", "business_address", "street_tokens", "domain_stem"] + FLAG_COLS)


def read_buckets(split, country):
    """Bucket rows of one (split, country), grouped by S1 in rank order, with candidate-side stats: the top
    CANDIDATES_PER_S1 e5 candidates, plus 02's extra candidates when present (and USE_EXTRA_CANDIDATES)."""
    path = BUCKET_DIR / f"{split}_{country}.parquet"
    if "e5_rank" in pq.read_schema(path).names:                      # 02 with extra sources: rank = position after filtering
        b = pq.read_table(path).to_pandas()
        e5 = b["e5_rank"].to_numpy()
        keep = (e5 < CANDIDATES_PER_S1) | (np.isnan(e5) & USE_EXTRA_CANDIDATES)
        b = b[keep].reset_index(drop=True)
        s1 = b["s1_entity_id"].to_numpy()
        starts = np.r_[0, np.flatnonzero(s1[1:] != s1[:-1]) + 1]
        b["rank"] = (np.arange(len(b)) - np.repeat(starts, np.diff(np.r_[starts, len(b)]))).astype(np.int16)
    else:
        b = pq.read_table(path, filters=[("rank", "<", CANDIDATES_PER_S1)]).to_pandas()
    s1 = b["s1_entity_id"].to_numpy()
    assert int((s1[1:] != s1[:-1]).sum()) + 1 == b["s1_entity_id"].nunique(), "bucket rows are not grouped by S1"
    # how the candidate scores against every S1 of the country that has it in its bucket
    codes, uniq = pd.factorize(b["candidate_entity_id"])
    score = b["score"].to_numpy(np.float32)
    order = np.lexsort((-score, codes))
    sc, ss = codes[order], score[order]
    start = np.r_[True, sc[1:] != sc[:-1]]
    r_sorted = np.arange(len(sc)) - np.maximum.accumulate(np.where(start, np.arange(len(sc)), 0))
    cand_rank = np.empty(len(b), np.float32)
    cand_rank[order] = r_sorted
    best = np.full(len(uniq), np.nan, np.float32)
    best[sc[start]] = ss[start]
    second = np.full(len(uniq), np.nan, np.float32)
    second[sc[r_sorted == 1]] = ss[r_sorted == 1]
    b["cand_n_s1"] = np.bincount(codes, minlength=len(uniq)).astype(np.float32)[codes]
    b["cand_rank"] = cand_rank
    b["cand_margin"] = np.where(cand_rank == 0, score - second[codes], score - best[codes]).astype(np.float32)
    return b


def chunk_bounds(b):
    """(start, end) row ranges of about PAIRS_PER_CHUNK pairs that never split an S1's bucket."""
    s1 = b["s1_entity_id"].to_numpy()
    starts = np.flatnonzero(np.r_[True, s1[1:] != s1[:-1]])
    idx = np.searchsorted(starts, np.arange(PAIRS_PER_CHUNK, len(b), PAIRS_PER_CHUNK))
    cuts = np.unique(np.r_[0, starts[idx[idx < len(starts)]], len(b)])
    return list(zip(cuts[:-1], cuts[1:]))


def sim(scorer, a, b):
    """rapidfuzz similarity of aligned string arrays; NaN where either side is empty."""
    if cpdist is not None:
        out = cpdist(list(a), list(b), scorer=scorer, workers=-1).astype(np.float32)
    else:
        out = np.array([scorer(x, y) for x, y in zip(a, b)], np.float32)
    out[(a == "") | (b == "")] = np.nan
    return out


def same(a, b):
    """1 when equal, 0 when different, NaN when either side is empty."""
    out = (a == b).astype(np.float32)
    out[(a == "") | (b == "")] = np.nan
    return out


def set_overlap(a, b):
    """Jaccard overlap and share of a's items found in b, for aligned arrays of frozensets (NaN when both empty)."""
    jac = np.full(len(a), np.nan, np.float32)
    cover = np.full(len(a), np.nan, np.float32)
    for i, (x, y) in enumerate(zip(a, b)):
        if x and y:
            inter = len(x & y)
            jac[i] = inter / len(x | y)
            cover[i] = inter / len(x)
        elif x or y:
            jac[i] = cover[i] = 0.0
    return jac, cover


def make_features(b, ent, stats):
    """float32 feature matrix (columns FEATURES) for the bucket rows b (whole S1 buckets, in rank order)."""
    qpos = ent.index.get_indexer(b["s1_entity_id"])
    cpos = ent.index.get_indexer(b["candidate_entity_id"])
    missing = int((qpos < 0).sum() + (cpos < 0).sum())
    assert missing == 0, f"{missing} bucket records are missing from processed/"

    def pair(col):
        v = ent[col].to_numpy()
        return v[qpos], v[cpos]

    f = {}
    by_s1 = b.groupby("s1_entity_id", sort=False)
    f["rank"] = b["rank"].to_numpy(np.float32)
    f["score"], f["cosine"] = b["score"].to_numpy(np.float32), b["cosine"].to_numpy(np.float32)
    f["s1_best_cosine"] = by_s1["cosine"].transform("max").to_numpy(np.float32)
    f["gap_to_best_score"] = (by_s1["score"].transform("max") - b["score"]).to_numpy(np.float32)
    f["gap_to_next_score"] = (b["score"] - by_s1["score"].shift(-1)).to_numpy(np.float32)
    f["s1_top5_cosine"] = b["s1_entity_id"].map(b[b["rank"] < 5].groupby("s1_entity_id")["cosine"].mean()).to_numpy(np.float32)
    for c in ("cand_n_s1", "cand_rank", "cand_margin"):
        f[c] = b[c].to_numpy(np.float32)
    x, y = pair("name_norm")
    f["name_ratio"], f["name_token_set"] = sim(fuzz.ratio, x, y), sim(fuzz.token_set_ratio, x, y)
    f["name_token_sort"], f["name_partial"] = sim(fuzz.token_sort_ratio, x, y), sim(fuzz.partial_ratio, x, y)
    f["s1_name_len"] = np.fromiter((len(v) for v in x), np.float32, len(x))
    f["cand_name_len"] = np.fromiter((len(v) for v in y), np.float32, len(y))
    x, y = pair("core_name")
    f["core_ratio"], f["core_jaro_winkler"] = sim(fuzz.ratio, x, y), sim(JaroWinkler.normalized_similarity, x, y)
    x, y = pair("name_lat")
    f["latin_ratio"], f["latin_token_set"] = sim(fuzz.ratio, x, y), sim(fuzz.token_set_ratio, x, y)
    x, y = pair("raw_name")
    f["raw_name_token_set"] = sim(fuzz.token_set_ratio, x, y)
    x, y = pair("compact")
    f["compact_ratio"], f["compact_partial"] = sim(fuzz.ratio, x, y), sim(fuzz.partial_ratio, x, y)
    f["phonetic_same"] = same(*pair("phonetic_code"))
    f["legal_form_same"] = same(*pair("legal_form"))
    x, y = pair("addr_norm")
    f["addr_ratio"], f["addr_token_set"] = sim(fuzz.ratio, x, y), sim(fuzz.token_set_ratio, x, y)
    x, y = pair("raw_addr")
    f["raw_addr_token_set"], f["raw_addr_partial"] = sim(fuzz.token_set_ratio, x, y), sim(fuzz.partial_ratio, x, y)
    f["cand_addr_len"] = np.fromiter((len(v) for v in y), np.float32, len(y))
    f["street_jaccard"], f["street_cover"] = set_overlap(*pair("street_set"))
    f["numbers_jaccard"], f["numbers_cover"] = set_overlap(*pair("numbers"))
    f["postcode_same"] = same(*pair("postcode"))
    f["house_no_same"] = same(*pair("house_no"))
    f["city_same"] = same(*pair("city"))
    f["state_same"] = same(*pair("state_code"))
    f["cand_is_s3"] = (b["candidate_source"].to_numpy() == "S3").astype(np.float32)
    for tag in TAGS:
        f[f"cand_{tag[4:]}"] = pair(tag)[1].astype(np.float32)
    if FEATURE_SET == "base":
        return np.column_stack([f[c] for c in FEATURES]).astype(np.float32)
    for kind in ("name", "addr"):
        w = weighted_overlap(*pair(f"{kind}_words"), stats[f"{kind}_idf"], stats["unseen_idf"])
        for j, c in enumerate(("jaccard", "cover", "miss_s1", "miss_cand")):
            f[f"{kind}_idf_{c}"] = w[j]
    x, y = pair("street_name")
    f["street_name_token_set"], f["street_name_ratio"] = sim(fuzz.token_set_ratio, x, y), sim(fuzz.ratio, x, y)
    f["house_number_same"] = same(*pair("house_number"))
    x, y = pair("name_key")
    freq, per = stats["name_freq"], 100_000 / stats["n"]             # S1 with this name per 100,000 S1 of the country
    f["s1_name_freq"] = np.log1p(per * np.fromiter((freq.get(v, 0) for v in x), np.float32, len(x)))
    f["cand_name_freq"] = np.log1p(per * np.fromiter((freq.get(v, 0) for v in y), np.float32, len(y)))
    for c in SOURCE_FEATURES:                                        # empty when 02 ran without extra sources
        f[c] = b[c].to_numpy(np.float32) if c in b.columns else np.full(len(b), np.nan, np.float32)
    return np.column_stack([f[c] for c in FEATURES]).astype(np.float32)


def featurize(b, ent, stats):
    """make_features over chunks of whole S1 buckets."""
    return np.vstack([make_features(b.iloc[s:e], ent, stats) for s, e in chunk_bounds(b)])


print(f"{len(FEATURES)} features")

## Ground truth
The true S1 of every train S2/S3 record, and the number of true matches of every train S1
(including matches that are not in its bucket, so they count as misses).

In [ ]:
gt = pd.read_csv(GT_FILE, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
GT_LIST = gt.set_index("source1_entity_id")["matched_entity_ids"]
pairs = gt[gt["matched_entity_ids"] != ""]
pairs = pairs.assign(entity_id=pairs["matched_entity_ids"].str.split(",")).explode("entity_id")
OWNER = pd.Series(pairs["source1_entity_id"].to_numpy(), index=pairs["entity_id"].to_numpy())
del gt, pairs
print(f"{len(GT_LIST):,} train S1, {int((GT_LIST == '').sum()):,} singletons, {len(OWNER):,} true pairs")

## Training pairs from the full-size train buckets
For each country: `TRAIN_S1_PER_COUNTRY` random train S1 with their buckets. The candidate-side
features were computed over every S1 of the country before the subset was taken.

In [ ]:
rng = np.random.default_rng(SEED)
X_parts, L_parts = [], []
for country in TRAIN_COUNTRIES:
    t0 = time.time()
    b = read_buckets("train", country)
    s1_all = b["s1_entity_id"].unique()
    chosen = rng.choice(s1_all, min(TRAIN_S1_PER_COUNTRY, len(s1_all)), replace=False)
    b = b[b["s1_entity_id"].isin(set(chosen))].reset_index(drop=True)
    ent = load_records("train", country, set(b["s1_entity_id"]) | set(b["candidate_entity_id"]))
    X_parts.append(featurize(b, ent, country_stats("train", country)))
    part = b[["s1_entity_id", "candidate_entity_id", "rank", "score", "cosine"]].copy()
    part["country"] = country
    part["is_true"] = part["candidate_entity_id"].map(OWNER).to_numpy() == part["s1_entity_id"].to_numpy()
    L_parts.append(part)
    print(f"train {country}: {len(s1_all):,} S1 in buckets, {len(chosen):,} used, {len(part):,} pairs "
          f"({int(part['is_true'].sum()):,} true), {time.time() - t0:.0f}s")
    del b, ent
    gc.collect()
X = np.vstack(X_parts)
L = pd.concat(L_parts, ignore_index=True)
y = L["is_true"].to_numpy()
del X_parts, L_parts
gc.collect()

# scored S1 = every training S1; true matches from the ground truth (bucket misses count)
EVAL_IDS = L["s1_entity_id"].unique()
TRUE_OF = {s: set(v.split(",")) if v else set() for s, v in GT_LIST.reindex(EVAL_IDS).fillna("").items()}
N_TRUE = pd.Series({s: len(v) for s, v in TRUE_OF.items()}, name="n_true").reindex(EVAL_IDS)
EVAL_COUNTRY = pd.Series(L.drop_duplicates("s1_entity_id").set_index("s1_entity_id")["country"]).reindex(EVAL_IDS).to_numpy()
L["s1_pos"] = L["s1_entity_id"].map(pd.Series(np.arange(len(EVAL_IDS)), index=EVAL_IDS)).to_numpy()
print(f"{len(L):,} training pairs, {len(EVAL_IDS):,} S1 ({int((N_TRUE == 0).sum()):,} singletons), "
      f"{X.nbytes / 1e9:.1f} GB of features, {time.time() - T0:.0f}s so far")

## Cross-validation grouped by S1
The training S1 are split into folds. Each fold is predicted by a model trained on the others,
which stops when the held-out fold stops improving.

In [ ]:
fold_of = pd.Series(rng.permutation(np.arange(len(EVAL_IDS)) % N_FOLDS), index=EVAL_IDS)
L["fold"] = L["s1_entity_id"].map(fold_of).to_numpy()


def fit(x_train, y_train, x_valid=None, y_valid=None, rounds=2000):
    """Train LightGBM; with a validation set, stop 50 rounds after its log-loss stops improving."""
    train_set = lgb.Dataset(x_train, y_train, feature_name=FEATURES)
    if x_valid is None:
        return lgb.train(PARAMS, train_set, rounds)
    valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)
    return lgb.train(PARAMS, train_set, rounds, valid_sets=[valid_set],
                     callbacks=[lgb.early_stopping(50, verbose=False)])


OOF = np.zeros(len(L), np.float32)
BEST_ITERS = []
for fold in range(N_FOLDS):
    t_fold = time.time()
    tr, va = L["fold"].to_numpy() != fold, L["fold"].to_numpy() == fold
    model = fit(X[tr], y[tr], X[va], y[va])
    OOF[va] = model.predict(X[va], num_iteration=model.best_iteration)
    BEST_ITERS.append(model.best_iteration)
    print(f"fold {fold}: {int(tr.sum()):,} train / {int(va.sum()):,} valid pairs, best iteration {model.best_iteration}, "
          f"valid log-loss {model.best_score['valid_0']['binary_logloss']:.4f}, {time.time() - t_fold:.0f}s")
L["p"] = OOF

## Threshold and cross-validation score
Same decision rule as notebook 03: keep p ≥ t, and give each S2/S3 record to at most one S1 (the
one with the highest p). The ceiling is a perfect matcher on the same buckets (the top
`CANDIDATES_PER_S1` e5 candidates plus 02's extra candidates). Here the ceiling is below 1 for two reasons: true matches outside the bucket, and the
full-size index.

In [ ]:
def score(df, keep):
    """Challenge macro F0.5 over the scored S1 when the rows of df selected by keep are the predictions."""
    pos = df["s1_pos"].to_numpy()
    n_pred = np.bincount(pos[keep], minlength=len(N_TRUE))
    tp = np.bincount(pos[keep], weights=df["is_true"].to_numpy()[keep].astype(float), minlength=len(N_TRUE))
    n_true = N_TRUE.to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        p, r = tp / n_pred, tp / n_true
        f = np.where(tp > 0, 1.25 * p * r / (0.25 * p + r), 0.0)
    f = np.where(n_true == 0, (n_pred == 0).astype(float), f)
    out = {"macro F0.5": f.mean(), "S1 with matches": f[n_true > 0].mean(), "singletons": f[n_true == 0].mean()}
    out.update({f"macro F0.5 {c}": f[EVAL_COUNTRY == c].mean() for c in sorted(set(EVAL_COUNTRY))})
    return out


def one_owner(df, p):
    """True for the rows where this S1 gives the candidate its highest p among all S1 in df."""
    order = np.argsort(-p, kind="stable")
    first = ~pd.Series(df["candidate_entity_id"].to_numpy()[order]).duplicated().to_numpy()
    owner = np.zeros(len(df), bool)
    owner[order[first]] = True
    return owner


def tune(df, p, extra, grid):
    """Best threshold on grid for keep = (p >= t) & extra, and its scores."""
    return max(((t, score(df, (p >= t) & extra)) for t in grid), key=lambda r: r[1]["macro F0.5"])


p, cos = L["p"].to_numpy(), L["cosine"].to_numpy()
owner_p, owner_cos = one_owner(L, p), one_owner(L, cos)
rows = [{"method": "predict nothing", "t": None, **score(L, np.zeros(len(L), bool))}]
t_cos, s_cos = tune(L, cos, owner_cos, np.round(np.arange(0.80, 0.995, 0.005), 3))
rows.append({"method": "cosine ≥ t + one owner (notebook 02 rule)", "t": t_cos, **s_cos})
t_plain, s_plain = tune(L, p, np.ones(len(L), bool), np.round(np.arange(0.02, 0.99, 0.01), 2))
rows.append({"method": "LightGBM p ≥ t", "t": t_plain, **s_plain})
T_BEST, s_best = tune(L, p, owner_p, np.round(np.arange(0.02, 0.99, 0.01), 2))
rows.append({"method": "LightGBM p ≥ t + one owner (used for the submission)", "t": T_BEST, **s_best})
rows.append({"method": "ceiling: perfect matcher on the buckets", "t": None,
             **score(L, L["is_true"].to_numpy())})
METRIC = pd.DataFrame(rows)
METRIC.to_csv(OUT_DIR / "model" / "metric_f05.csv", index=False)
L["kept"] = (p >= T_BEST) & owner_p
L[["s1_entity_id", "candidate_entity_id", "country", "rank", "cosine", "fold", "p", "is_true", "kept"]].to_parquet(
    OUT_DIR / "model" / "oof_pairs.parquet", index=False)

kept = L[L["kept"]]
reference = macro_f05(kept.groupby("s1_entity_id")["candidate_entity_id"].apply(list).to_dict(), TRUE_OF, EVAL_IDS)
assert abs(reference - s_best["macro F0.5"]) < 1e-9, (reference, s_best["macro F0.5"])
print(f"threshold t = {T_BEST}; vectorized score matches the reference metric")
METRIC.round(4)

### Score at other thresholds
A flat curve around t means the choice is not fragile.

In [ ]:
pd.DataFrame([{"t": t, **score(L, (p >= t) & owner_p)} for t in np.round(np.arange(0.1, 0.95, 0.05), 2)]) \
    .set_index("t")[["macro F0.5", "S1 with matches", "singletons"]].round(4)

## Transfer between countries (stand-in for France)
Train on one country, predict the other with the threshold above. A small drop means the model
does not rely on country-specific patterns.

In [ ]:
ROUNDS = max(50, int(np.mean(BEST_ITERS) * 1.1))
TRANSFER = []
if CROSS_COUNTRY_CHECK and len(TRAIN_COUNTRIES) > 1:
    for test_c in TRAIN_COUNTRIES:
        t_c = time.time()
        tr, te = L["country"].to_numpy() != test_c, L["country"].to_numpy() == test_c
        model = fit(X[tr], y[tr], rounds=ROUNDS)
        sub = L[te]
        p_te = model.predict(X[te]).astype(np.float32)
        s = score(sub, (p_te >= T_BEST) & one_owner(sub, p_te))[f"macro F0.5 {test_c}"]
        TRANSFER.append({"trained on": " + ".join(c for c in TRAIN_COUNTRIES if c != test_c), "scored on": test_c,
                         "macro F0.5": round(s, 4), "cross-validation (all countries)": round(s_best[f"macro F0.5 {test_c}"], 4),
                         "seconds": round(time.time() - t_c)})
pd.DataFrame(TRANSFER)

## Final model on all training pairs

In [ ]:
t_final = time.time()
FINAL = fit(X, y, rounds=ROUNDS)
FINAL.save_model(str(OUT_DIR / "model" / "lgbm_matcher.txt"))
IMPORTANCE = pd.DataFrame({"feature": FEATURES, "gain": FINAL.feature_importance("gain"),
                           "splits": FINAL.feature_importance("split")}).sort_values("gain", ascending=False)
IMPORTANCE["gain %"] = (100 * IMPORTANCE["gain"] / IMPORTANCE["gain"].sum()).round(2)
IMPORTANCE.to_csv(OUT_DIR / "model" / "feature_importance.csv", index=False)
TRAIN_RATES = L.groupby("country").apply(lambda g: pd.Series({
    "S1 with a predicted match (cv)": g.loc[g["kept"], "s1_entity_id"].nunique() / g["s1_entity_id"].nunique(),
    "S1 with a true match": float((N_TRUE.reindex(g["s1_entity_id"].unique()) > 0).mean()),
    "matches per matched S1 (cv)": g["kept"].sum() / max(g.loc[g["kept"], "s1_entity_id"].nunique(), 1)}))
del X
gc.collect()
print(f"final model: {ROUNDS} rounds in {time.time() - t_final:.0f}s")
IMPORTANCE[["feature", "gain %", "splits"]].head(15)

## Predict the test buckets
Per country: features in chunks, probability for every pair, one owner per S2/S3, keep p ≥ t.
The candidates of an S1 are exactly the pairs scored here, in rank order; its matches are the kept
ones, most likely first.

**Check France here.** France has no labels, so its score cannot be measured. Compare its
"S1 with a predicted match" and "matches per matched S1" with India and the US, and with the
training rates printed above them (about 94% and 3.5). Much higher values for France mean false
matches: in a local test on a thin 1% sample, France matched lookalikes (same city, house
number, a generic word and legal form, different street) about 3x as often as India/US. If that
shows up here, set a stricter cutoff, e.g. `COUNTRY_THRESHOLDS = {"France": 0.9}`, and re-run.

In [ ]:
def join_by_s1(s1, ids):
    """{s1: 'id,id,...'} for rows grouped by S1 (order kept, duplicates removed)."""
    if not len(s1):
        return {}
    starts = np.flatnonzero(np.r_[True, s1[1:] != s1[:-1]])
    ends = np.r_[starts[1:], len(s1)]
    return {s1[a]: ",".join(dict.fromkeys(ids[a:e])) for a, e in zip(starts, ends)}


CANDIDATES, MATCHES, TEST_STATS = {}, {}, []
for country in TEST_COUNTRIES:
    t0 = time.time()
    b = read_buckets("test", country)
    ent = load_records("test", country, set(b["s1_entity_id"]) | set(b["candidate_entity_id"]))
    stats = country_stats("test", country)
    prob = np.empty(len(b), np.float32)
    for s, e in chunk_bounds(b):
        prob[s:e] = FINAL.predict(make_features(b.iloc[s:e], ent, stats))
    t_country = COUNTRY_THRESHOLDS.get(country, T_BEST)
    keep = (prob >= t_country) & one_owner(b, prob)
    CANDIDATES.update(join_by_s1(b["s1_entity_id"].to_numpy(), b["candidate_entity_id"].to_numpy()))
    kept = pd.DataFrame({"s1": b["s1_entity_id"].to_numpy()[keep], "cand": b["candidate_entity_id"].to_numpy()[keep],
                         "p": prob[keep]}).sort_values(["s1", "p"], ascending=[True, False])
    MATCHES.update(join_by_s1(kept["s1"].to_numpy(), kept["cand"].to_numpy()))
    pd.DataFrame({"s1_entity_id": b["s1_entity_id"], "candidate_entity_id": b["candidate_entity_id"],
                  "rank": b["rank"], "p": prob}).to_parquet(OUT_DIR / "test_probabilities" / f"{country}.parquet", index=False)
    n_s1 = b["s1_entity_id"].nunique()
    TEST_STATS.append({"country": country, "t": t_country, "S1 with buckets": n_s1, "pairs scored": len(b),
                       "S1 with a predicted match": round(kept["s1"].nunique() / n_s1, 4),
                       "matches per matched S1": round(len(kept) / max(kept["s1"].nunique(), 1), 2),
                       "seconds": round(time.time() - t0)})
    print(f"test {country}: {n_s1:,} S1, {len(b):,} pairs, {len(kept):,} matches, {time.time() - t0:.0f}s")
    del b, ent, prob, keep, kept
    gc.collect()
print("training data, for comparison:")
display(TRAIN_RATES.round(3))
pd.DataFrame(TEST_STATS)

## Write the submission files
Every S1 in `test_source1.tsv` gets exactly one row, in file order, with an empty list when it
has no candidates or no match. The files are tab-separated, UTF-8 and use `\n` line endings,
with the headers from the problem statement.

In [ ]:
def read_ids(path):
    """First-column IDs of a TSV, read exactly as validate_submission.py reads them (file order, no duplicates)."""
    with open(path, encoding="utf-8") as f:
        next(f, None)
        return list(dict.fromkeys(line.split("\t", 1)[0].strip() for line in f if line.strip()))


def write_tsv(path, header, values, s1_ids):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\t".join(header) + "\n")
        for s in s1_ids:
            f.write(f"{s}\t{values.get(s, '')}\n")


TEST_S1 = read_ids(TEST_S1_FILE)
MATCHING_FILE = OUT_DIR / "output" / "matching_results.tsv"
CANDIDATE_FILE = OUT_DIR / "output" / "candidate_pairs.tsv"
write_tsv(MATCHING_FILE, ["source1_entity_id", "matched_entity_ids"], MATCHES, TEST_S1)
write_tsv(CANDIDATE_FILE, ["source1_entity_id", "candidate_entity_ids"], CANDIDATES, TEST_S1)
not_in_test = set(CANDIDATES) - set(TEST_S1)
print(f"{len(TEST_S1):,} test S1 written | with candidates: {sum(s in CANDIDATES for s in TEST_S1):,} | "
      f"with matches: {sum(s in MATCHES for s in TEST_S1):,} | bucket S1 not in test_source1.tsv: {len(not_in_test)}")
print(f"{MATCHING_FILE}: {MATCHING_FILE.stat().st_size / 1e6:.0f} MB | {CANDIDATE_FILE}: {CANDIDATE_FILE.stat().st_size / 1e6:.0f} MB")

CONFIG = {"features": FEATURES, "params": PARAMS, "rounds": ROUNDS, "fold_best_iterations": BEST_ITERS,
          "threshold": T_BEST, "country_thresholds": COUNTRY_THRESHOLDS, "one_owner": True,
          "candidates_per_s1": CANDIDATES_PER_S1,
          "train_s1_per_country": TRAIN_S1_PER_COUNTRY, "cv_scores": METRIC.to_dict("records"), "transfer": TRANSFER,
          "test": TEST_STATS, "buckets": str(BUCKET_DIR), "processed": str(PROCESSED_DIR),
          "minutes": round((time.time() - T0) / 60, 1)}
_ = (OUT_DIR / "model" / "matcher_config.json").write_text(json.dumps(CONFIG, indent=1, default=str), encoding="utf-8")

## Check the files against the submission rules
The same rules as `validate_submission.py`, including its optional ID-existence check: header,
one row per test S1 (no missing, extra or duplicate rows), no duplicate IDs in a list, only
S2-/S3- IDs, every ID exists in the test S2/S3 files, and matches ⊆ candidates. Then the official
validator runs too, if it is attached as an input (it is stdlib-only).

In [ ]:
def check_file(path, header, required, valid_ids):
    """Rule violations of one submission file, and its {s1: set(ids)} mapping."""
    problems, mapping, rows = [], {}, []
    with open(path, encoding="utf-8") as f:
        if f.readline().rstrip("\n").split("\t") != header:
            problems.append(f"header is not {header}")
        for line in f:
            s1, tab, rest = line.partition("\t")
            if not tab:
                problems.append(f"row without a tab: {line[:60]!r}")
                continue
            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            rows.append(s1)
            if len(ids) != len(set(ids)):
                problems.append(f"duplicate ID in the list of {s1}")
            mapping[s1] = set(ids)
    bad_prefix = {i for v in mapping.values() for i in v if not i.startswith(("S2-", "S3-"))}
    unknown = {i for v in mapping.values() for i in v if i not in valid_ids}
    for label, found in [("duplicate S1 rows", len(rows) - len(set(rows))), ("missing test S1", len(required - set(rows))),
                         ("S1 not in the test set", len(set(rows) - required)), ("IDs without S2-/S3- prefix", len(bad_prefix)),
                         ("IDs not in test S2/S3", len(unknown))]:
        if found:
            problems.append(f"{found:,} {label}")
    return problems, mapping


required = set(TEST_S1)
valid_ids = set(read_ids(TEST_DIR / "test_source2.tsv")) | set(read_ids(TEST_DIR / "test_source3.tsv"))
problems_m, matched = check_file(MATCHING_FILE, ["source1_entity_id", "matched_entity_ids"], required, valid_ids)
problems_c, candidates = check_file(CANDIDATE_FILE, ["source1_entity_id", "candidate_entity_ids"], required, valid_ids)
not_subset = sum(1 for s, v in matched.items() if v - candidates.get(s, set()))
print("matching_results.tsv:", problems_m or "OK")
print("candidate_pairs.tsv:", problems_c or "OK")
print("matches that are not candidates:", not_subset or "none")
assert not problems_m and not problems_c and not not_subset, "fix the problems above before submitting"
del valid_ids

if VALIDATOR is not None:
    result = subprocess.run([sys.executable, str(VALIDATOR), "--matching", str(MATCHING_FILE), "--candidate",
                             str(CANDIDATE_FILE), "--test-dir", str(TEST_DIR), "--check-ids"],
                            capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout[-4000:], result.stderr[-2000:])
else:
    print("validate_submission.py not attached: run it locally on the downloaded files (see the notebook text)")
print(f"done in {(time.time() - T0) / 60:.1f} min")